# N075 · 最近公共祖先与返回值语义

**目标：** 用子树是否包含目标定义递归返回信息。

**先修：** N071, N073。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 普通树LCA；BST路径；节点身份；目标存在性；路径法对照。

**配套讲解来源：** [同名逐章意见](../../comment/075_lowest_common_ancestor.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

给你一棵树和两个节点 p、q，问：p 和 q 的**最近公共祖先（LCA）**是谁？直观地说，就是"同时是 p 的祖先和 q 的祖先、而且离它们最近的那个节点"——你可以想象两个人分别从 p 和 q 往上爬，爬到第一个碰面的地方就是 LCA。本章给出两个实现：`lowest_common_ancestor` 适用任何二叉树（节点值可以重复），`lca_bst` 专门吃二叉搜索树（更快，但要求两个目标真的在树里）。

举一个具体到数字的例子。输入这棵树（层序序列 `[6,2,8,0,4,7,9,None,None,3,5]`，也是 “运行示例”部分/“自动测试”部分 共用的例子树）：

```
            6
          /   \
         2     8
        / \   / \
       0   4 7   9
          / \
         3   5
```

问 p=0、q=5 的 LCA：从 0 往上爬是 0→2→6，从 5 往上爬是 5→4→2→6，两条路在 2 第一次碰面，所以答案是节点 2。为什么不是 6？6 也是公共祖先，但它比 2 离 p、q 更远；"最近"就是要挑碰面点里最深的那个。再问 p=3、q=7：路径 3→4→2→6 和 7→8→6 只在 6 相交，答案是 6。

把两组查询的"根路径"并排写出来更好对照（路径按从根到节点写）：

| 查询 | 根到 p 的路径 | 根到 q 的路径 | 共同前缀的最后节点（LCA） |
|------|---------------|---------------|---------------------------|
| p=0, q=5 | 6, 2, 0 | 6, 2, 4, 5 | 2 |
| p=3, q=7 | 6, 2, 4, 3 | 6, 8, 7 | 6 |

这两行也就是"路径法对照"的完整演算：从根开始逐位比，最后一个相同的位置就是答案。

顺带明确两个边界约定：p 和 q 是同一个节点时，LCA 就是它自己（两条路径完全重合）；p、q 之一是根时，LCA 至多是根、也可能是更深的节点，取决于另一条路径怎么走。本章两个实现对这些边界都自然成立，不需要特判。

## 2. 基础知识：先读懂这些词

- **最近公共祖先（LCA）**：同时位于"根到 p"和"根到 q"两条路径上、且深度最大的节点。允许 p 自己是 q 的祖先（比如 LCA(2,4) 就是 2 自己，因为 2 在"根到 4"的路径上）。
- **祖先（ancestor）**：从某节点一路往上走到根（含它自己、含根）经过的所有节点。注意这个约定把"节点自己"也算进自己的祖先链——没有这条约定，"p 是 q 的祖先时 LCA 是 p"就说不通了。
- **父指针映射（parent map）**：树本来只有从上到下的孩子指针，我们用一次遍历把"每个节点的父亲是谁"记进字典，之后就能反复"往上爬"。这是把树问题转化成链问题的常用手法。
- **祖先集合**：把 p 一路向上的所有节点（含 p 自己）装进一个集合。之后任何节点想知道"我是不是 p 的祖先"，一次集合查询就够。注意这里存的是**节点对象**而不是值，原因见下一条。
- **节点身份（identity）与值（value）的区别**：两个 `TreeNode(2)` 是两个不同的对象，值相同不代表是同一个节点。判断"在不在树里""是不是同一个祖先"必须用对象身份（Python 的 `is` / 哈希），用值比较会在"树里有重复值"时张冠李戴。本章一般树版本正是靠比较身份，才允许树中存在相同值的节点。
- **目标存在性**：调用者传进来的 p、q 可能根本不在树里。本章一般树版本明确约定此时返回 `None`；而 BST 版本不做这个检查，它默认两个目标都在树里——这个前提差异是本章标题里"返回值语义"的一部分。
- **BST 的分叉定位**：BST 里如果 p、q 的键一个比当前节点小、一个比大（或者有一个等于当前节点），当前节点就是分叉点，即 LCA；如果两个键都在同一侧，那 LCA 必然也在那一侧的子树里，往那边走一步再判断。
- **路径法对照（暴力基准）**：最朴素也最不会错的求 LCA 办法——分别列出"根到 p"和"根到 q"两条完整路径，两条路径从根开始必然开头一段相同（至少共享根），分岔前的最后一个相同节点就是 LCA。它要多花 O(n) 空间，但逻辑透明，所以本章测试拿它当"标准答案"来对拍两个实现。
- **返回值语义**：函数返回的到底是什么？本章两个实现返回的都是树里**真实存在的那个节点对象**（测试用 `is` 验证），而不是一个值相等的新节点；目标缺失时一般树版本明确返回 `None`。写接口前把"返回什么、缺失时返回什么"说清楚，就是"返回值语义"。

## 3. 思路推导：从小例子开始

### 1. 一般树：父指针 + 祖先集合

- **Step 1**：先花一次遍历把全树的父指针记下来。从根出发做深度优先，每访问一个节点，就把它的两个孩子（如果有）的父亲登记成自己。字典初始 `{root: None}`，根没有父亲。
- **Step 2**：爬 p 的祖先链，把沿途所有节点（含 p）塞进集合 `ancestors`。
- **Step 3**：爬 q 的祖先链，第一次碰到"在 p 祖先集合里"的节点就是答案。
- **Step 4**：先把例子树的父指针表整个列出来（每行是"节点 → 它的父亲"）：2→6、8→6、0→2、4→2、7→8、9→8、3→4、5→4（根 6 的父亲是 None）。有了这张表，"往上爬"就是查表换人。
- **Step 5**：手算 p=7、q=3：从 7 爬是 7→8→6，建集合 {7, 8, 6}；从 3 爬：3 不在集合里，往上 4 不在，往上 2 不在，往上 6 在——返回 6。再手算 “运行示例”部分 那组 p=2、q=4：从 2 爬建集合 {2, 6}；从 4 爬：4 不在，往上 2 在——返回 2。注意后一个例子里答案就是 p 自己，因为 p 在"根到 q"的路上，公共祖先"最近"可以近到和 p 重合。
- **Step 6**：目标缺失处理。Step 1 结束时检查 p、q 是否都被记进过父指针字典；没记进去说明它根本不是树里的节点，直接返回 `None`。比如传一个新造的 `TreeNode(2)`（值和树里的 2 相同但对象不同），它在字典里查不到，函数如实返回 `None`。

### 2. BST：按键分叉，一路向下

- **Step 7**：先把两个目标的键排好序，记 `low,high`。从根开始向下走。
- **Step 8**：在当前节点判断三类情况：当前键比 low 还小，说明两个目标都在右边，走右孩子；当前键比 high 还大，两个目标都在左边，走左孩子；否则当前键卡在 `[low, high]` 之间（含等于），p、q 分居两侧（或其一就是当前节点），当前节点就是 LCA，返回。
- **Step 9**：手算例子树上 p=0、q=5：low=0、high=5。节点 6：6>5，走左；节点 2：0≤2≤5，分叉成立，返回 2。再算 p=3、q=9：low=3、high=9。节点 6 在区间内？3≤6≤9，成立，直接返回 6——和第一节的结论一致。
- **Step 10**：`while node` 走到空还没分叉（理论上只在不满足"目标都在树里"前提时发生），返回 `None`。

“运行示例”部分 用 p=2、q=4 验证两个实现答案一致：一般树法爬 4 的链 4→2 立刻命中集合 {2,6}，返回 2；BST 法在节点 6 处 6>4 走左，到节点 2 处 2 恰好落在 [2,4] 区间内，也返回 2。

## 4. 核心代码：lowest_common_ancestor

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def lowest_common_ancestor(root, p, q):
    if root is None:
        return None
    parent = {root: None}
    stack = [root]
    while stack and (p not in parent or q not in parent):
        node = stack.pop()
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)
    if p not in parent or q not in parent:
        return None
    ancestors = set()
    node = p
    while node is not None:
        ancestors.add(node)
        node = parent[node]
    node = q
    while node not in ancestors:
        node = parent[node]
    return node
```

### 逐段读懂代码

`TreeNode` 和 `tree_from_level` 与前几章相同，直接看两个新函数。

### 1. 一般树版本

（对应程序见下方分段实现与完整代码。）

逐段读：

- `if root is None: return None`：空树没有祖先可言。
- 第一个 `while` 建父指针表。循环条件 `p not in parent or q not in parent` 很讲究：一旦 p、q 都被登记过了就没必要继续扫，提前收工；如果扫完整棵树（栈空）还差谁没找到，循环同样结束，交给下一行判决。
- 第二个 `if` 是存在性判决：p 或 q 没出现在父指针字典里，等于"这个节点不是树的一分子"，返回 `None`。
- 接着把 p 的祖先链装进集合：`node=p`，只要 `node` 不是 `None`（爬过了根）就加进集合、换成父亲。根的父亲是 `None`，循环自然停在根之后，集合里含 p 自己到根的所有节点。
- 最后从 q 往上爬：`while node not in ancestors` 一直换父亲，第一次落进集合就停下——爬不动而 `node` 变成 `None` 的情况不会发生，因为 p、q 都在树里，根必然同时是两者的祖先，最坏在根碰面。返回这个碰面点。

### 2. BST 版本

```python
def lca_bst(root, p, q):
    low, high = sorted((p.val, q.val))
    node = root
    while node:
        if node.val < low:
            node = node.right
        elif node.val > high:
            node = node.left
        else:
            return node
    return None
```

- `low,high=sorted((p.val,q.val))` 一行完成"取两键中较小者当 low、较大者当 high"。
- 循环体就是第三节 Step 7 的三分支：小于 low 全在右、大于 high 全在左、夹在中间即分叉返回。
- 没有 `p`、`q` 是否在树里的检查——这是它比一般树版本快的原因，也是它的前提（“思路推导”部分 明确写了"要求两目标确实属于树"）。如果目标缺失，它会走到"假如目标存在会分叉的位置"返回那个节点，答案未必有意义。
- 辅助空间只有三个变量，不建任何字典或集合。

两个实现放在一起对比，各自的定位就清楚了：

| 维度 | `lowest_common_ancestor`（一般树） | `lca_bst`（BST） |
|------|-------------------------------------|------------------|
| 适用树 | 任何二叉树，允许重复值 | 必须是 BST，键互异 |
| 时间 | O(n)（全树铺父指针表） | O(h)（一路向下） |
| 辅助空间 | O(n)（字典 + 集合） | O(1) |
| 目标缺失 | 返回 `None` | 不检查，答案可能无意义 |

选择口诀：树不是 BST，或者值有重复、目标可能不在树里，用一般树版本；树是干净的 BST 且目标保证在树里，用 BST 版本享受 O(h) 时间和 O(1) 空间。

## 5. 分段实现：按顺序运行

**本章接口：** `lowest_common_ancestor(root, p, q)`、`lca_bst(root, p, q)`

### TreeNode

In [1]:
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

### tree_from_level

In [2]:
def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    return root

### lowest_common_ancestor

In [3]:
def lowest_common_ancestor(root, p, q):
    if root is None:
        return None
    parent = {root: None}
    stack = [root]
    while stack and (p not in parent or q not in parent):
        node = stack.pop()
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)
    if p not in parent or q not in parent:
        return None
    ancestors = set()
    node = p
    while node is not None:
        ancestors.add(node)
        node = parent[node]
    node = q
    while node not in ancestors:
        node = parent[node]
    return node

### lca_bst

In [4]:
def lca_bst(root, p, q):
    low, high = sorted((p.val, q.val))
    node = root
    while node:
        if node.val < low:
            node = node.right
        elif node.val > high:
            node = node.left
        else:
            return node
    return None

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([6,2,8,0,4,7,9,None,None,3,5])
p=r.left; q=r.left.right
show_table(['p','q','一般树LCA','BST LCA'],[(p.val,q.val,lowest_common_ancestor(r,p,q).val,lca_bst(r,p,q).val)])

p,q,一般树LCA,BST LCA
2,4,2,2


## 7. 正确性、适用条件与复杂度

从q向上首次进入p的祖先集合，不可能有更低的公共祖先；所有公共祖先都在该节点以上。BST两目标位于同侧时祖先必在该侧，否则当前节点是最低分叉。

**代价：** 一般树O(n)时间空间；BST单次O(h)时间/O(1)辅助空间。比较身份而非值，一般树允许相同值节点。

### 展开理解

**一般树版本为什么对？** p 的祖先集合恰好是"根到 p 路径"上的所有节点。从 q 一路向上，第一次撞进这个集合的节点 x，同时是 p 和 q 的祖先（前者因为 x 在集合里，后者因为 q 是从 x 之下爬上来的）；而在 x 下方的 q 的祖先都不在集合里，不可能是公共祖先；q 的所有更上方的祖先虽然也是 p 的祖先，但都比 x 高（离 p、q 更远）。所以 x 就是最深的那个公共祖先，正是定义要的答案。中间的父指针表是这一切的地基：它让"向上爬"成为可能，又因为登记的是对象身份，树里即使有重复值也不会认错人。

**BST 版本为什么对？** 每一步判断都在用 BST 的规矩：如果当前节点的键比两个目标的键都小，两个目标只能藏在右子树，它们的一切公共祖先（包括 LCA）也必然在右子树里，往右走一步不会错过答案；比两个都大时同理往左。一旦当前键把 `[low,high]` 夹住（某个目标在左、另一个在右，或目标之一就是当前节点），那么当前节点同时出现在"根到 p"和"根到 q"两条路上，而它的任何后代都不可能同时照顾到两侧——它就是分叉点，也是 LCA。

**两个常见疑问。** 第一问：p 自己是 q 的祖先时（比如 LCA(2,4)=2），算法还成立吗？成立。p 的祖先集合包含 p 自己，从 q 往上爬最先撞到的就是 p——"最近的公共祖先"允许近到和 p 重合，这是定义的一部分。第二问：为什么一般树版本敢允许重复值？因为父指针字典和祖先集合的键都是节点对象（按内存身份哈希），树里就算有两个值都叫 2 的节点，它们也是两个不同的键，爬链时绝不会认错人；换成语值做键，这条保证就没了。

**复杂度**：一般树版本要建整棵树的父指针表，时间和空间都是 O(n)；n=10⁵ 的树就是十万级的一次扫描加一张十万条目的字典，仍然是一瞬间的事。BST 版本每步只走一层、只用三个变量，时间 O(h)、辅助空间 O(1)：随机树高约 log₂n，n=10⁵ 时大约 17 步；最坏退化成链则是 10⁵ 步。一句话总结差距：一般树版本"先铺路再爬"，BST 版本"顺着路标直接走"。

## 8. 单元测试：每个用例在检查什么

- `assert lowest_common_ancestor(r,r.left,r.right) is r`：p=2、q=8，LCA 是根 6。注意断言用的是 `is` 而不是 `==`——必须返回**同一个对象**才算对，这本身就是"身份语义"的体现。
- `assert lowest_common_ancestor(r,r.left,r.left.right) is r.left`：p=2、q=4，其中 p 是 q 的祖先，答案是 p 自己。它测的是"祖先包含自身"这个容易被漏掉的约定。
- `assert lowest_common_ancestor(r,TreeNode(2),r.left) is None`：p 是一个值等于 2 的**新造节点**，不在树里；哪怕值撞车，身份也对不上，必须返回 `None`。这条专测"目标缺失"和"值相同不等于同一节点"两件事。
- 后面的双重循环对拍是重头戏：先独立铺一份父指针表，写一个"路径法"笨答案——`path(n)` 列出节点 n 到根的完整路径，两条路径从根开始逐位比对（`zip` + `a is b`），最后一个相同的位置就是 LCA。拿一对节点演算一遍：p=4 的路径是 [6, 2, 4]，q=9 的路径是 [6, 8, 9]；zip 逐位比：6 和 6 是同一个对象（True），2 和 8 不是（False），4 和 9 不是（False）；`common` 收集到 [6]，`common[-1]` 就是 6。然后对树里**所有节点对**（10 个节点共 100 对）逐一断言：两个实现都必须和路径法给出同一个对象。所有对的覆盖让"p 是 q 祖先""分居两树""同侧深层"各种位置关系全部被测到。顺带注意 `path` 里 `out[::-1]` 这一步：父指针爬出来的是"从节点到根"，要翻转成"从根到节点"，两条路径的公共前缀才对得上号。

最后一行打印 `N075: 所有本章断言通过`，表示快照、身份、双重对拍全部过关。

In [6]:
r = tree_from_level([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])

assert lowest_common_ancestor(r, r.left, r.right) is r

assert lowest_common_ancestor(r, r.left, r.left.right) is r.left

assert lowest_common_ancestor(r, TreeNode(2), r.left) is None

nodes = []

parent = {r: None}

stack = [r]

while stack:
    n = stack.pop()
    nodes.append(n)
    for c in (n.left, n.right):
        if c:
            parent[c] = n
            stack.append(c)

def path(n):
    out = []
    while n:
        out.append(n)
        n = parent[n]
    return out[::-1]

for p in nodes:
    for q in nodes:
        common = [a for a, b in zip(path(p), path(q)) if a is b]
        assert lowest_common_ancestor(r, p, q) is common[-1]
        assert lca_bst(r, p, q) is common[-1]

print('N075: 所有本章断言通过')

N075: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 扩展到目标节点可能缺失的自编任务。

**练习 2：** 解释值相同不表示同一节点。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（目标可能缺失的扩展）**：提示——一般树版本已经返回 `None`，真正的缺口在 BST 版本：`lca_bst` 从不检查目标在不在树里。想清楚它对缺失目标会返回什么（提示：走到"假如存在会在哪分叉"的位置就返回，这个节点可能根本不是任何目标的祖先）。手算起点：在例子树上问 `lca_bst(r, TreeNode(100), r.left)`：low=2、high=100，根 6 落在区间内，直接返回 6——可 100 根本不在树里，6 也不是 100 的祖先，这个答案纯属碰巧。修复方向有两个：一是先各做一次 O(h) 的存在性查找再进主循环；二是在主循环里顺路验证"p、q 确实能沿当前方向找到"。设计约定（缺失返回 `None` 还是抛异常）并写明边界：p 缺、q 缺、两个都缺。
- **练习 2（值相同不表示同一节点）**：提示——第三条断言就是现成素材。两个 `TreeNode(2)` 的 `val` 都是 2，`==` 比较值相等，但它们是两块独立的内存对象。设计实验：建一棵有两个相同值的树（比如 `[2,2,3]`，根和左孩子都叫 2），分别用"值比较"和"身份比较"实现 LCA，构造一对目标（比如 p=左边的 2、q=3）让两种实现给出不同答案，用这个反例说明为什么父指针字典和祖先集合必须存对象。也可以顺手在解释器里验证：`TreeNode(2) in {TreeNode(2)}` 是 `False`（集合按对象哈希），而 `TreeNode(2).val == TreeNode(2).val` 是 `True`——两个事实并排放，区别一目了然。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    return root

def lowest_common_ancestor(root, p, q):
    if root is None:
        return None
    parent = {root: None}
    stack = [root]
    while stack and (p not in parent or q not in parent):
        node = stack.pop()
        for child in (node.left, node.right):
            if child is not None:
                parent[child] = node
                stack.append(child)
    if p not in parent or q not in parent:
        return None
    ancestors = set()
    node = p
    while node is not None:
        ancestors.add(node)
        node = parent[node]
    node = q
    while node not in ancestors:
        node = parent[node]
    return node

def lca_bst(root, p, q):
    low, high = sorted((p.val, q.val))
    node = root
    while node:
        if node.val < low:
            node = node.right
        elif node.val > high:
            node = node.left
        else:
            return node
    return None

# 示例与回归测试
r = tree_from_level([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])

assert lowest_common_ancestor(r, r.left, r.right) is r

assert lowest_common_ancestor(r, r.left, r.left.right) is r.left

assert lowest_common_ancestor(r, TreeNode(2), r.left) is None

nodes = []

parent = {r: None}

stack = [r]

while stack:
    n = stack.pop()
    nodes.append(n)
    for c in (n.left, n.right):
        if c:
            parent[c] = n
            stack.append(c)

def path(n):
    out = []
    while n:
        out.append(n)
        n = parent[n]
    return out[::-1]

for p in nodes:
    for q in nodes:
        common = [a for a, b in zip(path(p), path(q)) if a is b]
        assert lowest_common_ancestor(r, p, q) is common[-1]
        assert lca_bst(r, p, q) is common[-1]

print('N075: 所有本章断言通过')

N075: 所有本章断言通过


## 11. 代表题与迁移

- **236. Lowest Common Ancestor of a Binary Tree**：普通树 LCA，直接对应 `lowest_common_ancestor`，本章给的是"父指针 + 祖先集合"的迭代解法（官方题解里更常见的是递归返回值解法，两者都要会）。
- **235. Lowest Common Ancestor of a Binary Search Tree**：BST 上的 LCA，直接对应 `lca_bst`，练的是"利用有序性把 O(n) 的搜索砍成 O(h) 的一路下行"，注意该题保证节点都在树中的前提。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。